# Credit Card Fraud Detection

This notebook aims to predict fraudulent credit card transactions using machine learning. We will use the provided dataset to train a model and evaluate its performance using the ROC AUC score.

**Update:** Previous models (Logistic Regression, HistGradientBoosting) were detecting 0 fraud cases. We are switching to a **ProbaBoost** approach which ensembles conditional probability classifiers on bootstrapped samples.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve, f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from joblib import Parallel, delayed
import gc

import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')

In [ ]:
train_df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Check for missing values
print(train_df.isnull().sum().sum())
print(test_df.isnull().sum().sum())

In [ ]:
# Class distribution
plt.figure(figsize=(8, 6))
sns.countplot(x='IsFraud', data=train_df)
plt.title('Class Distribution')
plt.savefig('class_distribution.png')
plt.show()

print(train_df['IsFraud'].value_counts(normalize=True))

The dataset is likely highly imbalanced.

In [ ]:
# Time and Amount distribution
fig, ax = plt.subplots(1, 2, figsize=(18,4))

amount_val = train_df['Transaction_Amount'].values
time_val = train_df['Time'].values

sns.distplot(amount_val, ax=ax[0], color='r')
ax[0].set_title('Distribution of Transaction Amount', fontsize=14)
ax[0].set_xlim([min(amount_val), max(amount_val)])

sns.distplot(time_val, ax=ax[1], color='b')
ax[1].set_title('Distribution of Transaction Time', fontsize=14)
ax[1].set_xlim([min(time_val), max(time_val)])

plt.savefig('feature_distribution.png')
plt.show()

In [ ]:
# Correlation Matrix
plt.figure(figsize=(20, 10))
corr = train_df.corr()
sns.heatmap(corr, cmap='coolwarm_r', annot_kws={'size':20})
plt.title('Correlation Matrix', fontsize=14)
plt.savefig('correlation_matrix.png')
plt.show()

## 3. Preprocessing

In [ ]:
# Scale Time and Amount
scaler = RobustScaler()

train_df['scaled_amount'] = scaler.fit_transform(train_df['Transaction_Amount'].values.reshape(-1,1))
train_df['scaled_time'] = scaler.fit_transform(train_df['Time'].values.reshape(-1,1))

test_df['scaled_amount'] = scaler.transform(test_df['Transaction_Amount'].values.reshape(-1,1))
test_df['scaled_time'] = scaler.transform(test_df['Time'].values.reshape(-1,1))

train_df.drop(['Time','Transaction_Amount'], axis=1, inplace=True)
test_df.drop(['Time','Transaction_Amount'], axis=1, inplace=True)

In [ ]:
# Separate features and target
X = train_df.drop(['IsFraud', 'id'], axis=1)
y = train_df['IsFraud']

X_test = test_df.drop(['id'], axis=1)

## 4. Model Training

In [ ]:
# Split for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### ProbaBoost Implementation (NEW)
Implementing the ProbaBoost algorithm to improve fraud detection.

In [ ]:
class ConditionProbabilitiesClassifier:
    def __init__(self, df, cat_feats, num_feats, target, nb_buckets=10):
        self.df = df
        self.cat_feats = cat_feats
        self.num_feats = num_feats
        self.target = target
        self.nb_buckets = nb_buckets
        self.cat_cond_probs = {}
        self.num_cond_probs = {}
        self.num_buckets = {}

    def fit(self):
        for col in self.cat_feats:
            self.cat_cond_probs[col] = self.df.groupby(col)[self.target].mean().to_dict()

        for col in self.num_feats:
            self.df[f'{col}_bucket'], bins = pd.qcut(self.df[col], self.nb_buckets, duplicates='drop', retbins=True, labels=False)
            self.num_buckets[col] = bins
            self.num_cond_probs[col] = self.df.groupby(f'{col}_bucket')[self.target].mean().to_dict()

    def predict_buckets(self, df, col):
        bins = self.num_buckets[col]
        # Use cut with the saved bins
        # We need to handle values outside the range by clipping or extending bins slightly
        # For simplicity, we can use pd.cut and fill na
        # A robust way is to use searchsorted or similar, but pd.cut is easiest
        # Note: pd.qcut returns categories, we need to map them to the mean target
        # The bins from qcut are edges. 
        
        # Adjust bins to include -inf and inf to handle outliers in test
        bins[0] = -np.inf
        bins[-1] = np.inf
        
        return pd.cut(df[col], bins=bins, labels=False, include_lowest=True).fillna(0).astype(int)

    def predict(self, df):
        pred_df = df.copy()
        
        # Categorical
        for col in self.cat_feats:
            # Map probabilities, fill unknown with global mean or 0
            global_mean = self.df[self.target].mean()
            pred_df[col] = pred_df[col].map(self.cat_cond_probs[col]).fillna(global_mean)

        # Numerical
        for col in self.num_feats:
            # Bucketize
            buckets = self.predict_buckets(pred_df, col)
            # Map probabilities
            # If a bucket was not seen in training (unlikely with qcut but possible if duplicates dropped bins), fill with mean
            global_mean = self.df[self.target].mean()
            
            # We need to map the bucket index to the probability
            # The num_cond_probs[col] keys are integers (bucket indices)
            pred_df[col] = buckets.map(self.num_cond_probs[col]).fillna(global_mean)

        # Average probabilities across features
        # This is a naive bayes-like approach but averaging probabilities
        preds = pred_df[self.cat_feats + self.num_feats].mean(axis=1)
        return preds

class ProbaBoost:
    def __init__(self, df, cat_feats, num_feats, target, nb_boosters=10, nb_buckets=10, random_state=42):
        self.df = df
        self.cat_feats = cat_feats
        self.num_feats = num_feats
        self.target = target
        self.nb_boosters = nb_boosters
        self.nb_buckets = nb_buckets
        self.random_state = random_state
        self.classifiers = []

    def train_booster(self, i):
        # Bootstrap sample
        sample_df = self.df.sample(len(self.df), replace=True, random_state=self.random_state + i)
        clf = ConditionProbabilitiesClassifier(sample_df, self.cat_feats, self.num_feats, self.target, self.nb_buckets)
        clf.fit()
        return clf

    def fit(self):
        self.classifiers = Parallel(n_jobs=-1)(delayed(self.train_booster)(i) for i in range(self.nb_boosters))

    def predict(self, df):
        # Average predictions from all boosters
        preds_list = Parallel(n_jobs=-1)(delayed(clf.predict)(df) for clf in self.classifiers)
        return np.mean(preds_list, axis=0)


In [ ]:
# Prepare data for ProbaBoost
# We will use the X_train and y_train combined
train_data = X_train.copy()
train_data['IsFraud'] = y_train

num_cols = X_train.columns.tolist()
cat_cols = [] # No explicit categorical columns in this dataset (all are float/int)

print("Training ProbaBoost...")
proba_boost = ProbaBoost(
    df=train_data,
    cat_feats=cat_cols,
    num_feats=num_cols,
    target='IsFraud',
    nb_boosters=30,
    nb_buckets=20
)
proba_boost.fit()
print("Training complete.")

In [ ]:
# Evaluate on Validation Set
y_pred_proba = proba_boost.predict(X_val)
print('ProbaBoost ROC AUC:', roc_auc_score(y_val, y_pred_proba))

# Optimize Threshold
# Since probabilities are very low, we need a fine-grained search
max_prob = y_pred_proba.max()
thresholds = np.linspace(0, max_prob, 100)
f1_scores = []
for thresh in thresholds:
    y_pred_binary = (y_pred_proba > thresh).astype(int)
    f1_scores.append(f1_score(y_val, y_pred_binary))

best_thresh = thresholds[np.argmax(f1_scores)]
print(f"Best Threshold (F1 Score): {best_thresh}")
print(f"Max F1 Score: {max(f1_scores)}")

### Old Models (Commented Out)
The following models were commented out because they were reportedly detecting 0 fraud cases.

In [ ]:
# Logistic Regression Baseline
# lr = LogisticRegression(class_weight='balanced', max_iter=1000)
# lr.fit(X_train, y_train)
# y_pred_lr = lr.predict_proba(X_val)[:,1]
# print('Logistic Regression ROC AUC:', roc_auc_score(y_val, y_pred_lr))

In [ ]:
# HistGradientBoosting Classifier
# hgb_clf = HistGradientBoostingClassifier(learning_rate=0.05, 
#                                          max_iter=100, 
#                                          max_depth=6, 
#                                          random_state=42)

# hgb_clf.fit(X_train, y_train)
# y_pred_hgb = hgb_clf.predict_proba(X_val)[:,1]
# print('HistGradientBoosting ROC AUC:', roc_auc_score(y_val, y_pred_hgb))

### Evaluation Plots

In [ ]:
# ROC Curve
fpr, tpr, thresholds_roc = roc_curve(y_val, y_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label='ProbaBoost (AUC = %0.4f)' % roc_auc_score(y_val, y_pred_proba))
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.savefig('roc_curve.png')
plt.show()

In [ ]:
# Confusion Matrix with Best Threshold
y_pred_binary = (y_pred_proba > best_thresh).astype(int)
cm = confusion_matrix(y_val, y_pred_binary)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix (Threshold = {best_thresh:.2f})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('confusion_matrix.png')
plt.show()

In [ ]:
# Feature Importance (Permutation Importance)
# Note: Permutation importance might be slow with the custom class and Parallel execution inside predict.
# Skipping for now or we can implement a simpler version if needed.
# result = permutation_importance(proba_boost, X_val, y_val, n_repeats=5, random_state=42, n_jobs=-1)
# sorted_idx = result.importances_mean.argsort()
# plt.figure(figsize=(10, 8))
# plt.boxplot(result.importances[sorted_idx].T, vert=False, labels=X_val.columns[sorted_idx])
# plt.title("Permutation Importance (test set)")
# plt.tight_layout()
# plt.savefig('feature_importance.png')
# plt.show()

## 5. Submission

In [ ]:
# Predict on test set
test_preds = proba_boost.predict(X_test)

submission = pd.DataFrame({'id': test_df['id'], 'IsFraud': test_preds})
submission.head()

In [ ]:
submission.to_csv('submission.csv', index=False)
print('Submission file saved!')